In [ ]:
#imports and installation
!pip install torch transformers peft bitsandbytes accelerate datasets

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model
from datasets import load_dataset
import bitsandbytes as bnb
import numpy as np

In [ ]:
#loading & processing dataset

ds = load_dataset("BuildaByte/Meditation-miniset-v0.2")["train"]

#getting unique prompts (must split by unique prompts or else duplicates exist)

unique_prompts = list(set(ds["user_prompt"]))
np.random.seed(42)
np.random.shuffle(unique_prompts)

n = len(unique_prompts)
train_prompts = set(unique_prompts[: int(n * 0.80)])
val_prompts = set(unique_prompts[int(n * 0.80): int(n * 0.90)])
test_prompts = set(unique_prompts[int(n * 0.90):])

#splitting rows based on which bucket their prompt falls into

train_ds = ds.filter(lambda x: x["user_prompt"] in train_prompts)
val_ds = ds.filter(lambda x: x["user_prompt"] in val_prompts)
test_ds = ds.filter(lambda x: x["user_prompt"] in test_prompts)

print(f"Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")

In [ ]:
#loading base model and QLoRA setup
model_name = "microsoft/Phi-3-mini-4k-instruct"

print(f"Starting to load the model {model_name}...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    load_in_4bit=True, #enables 4-bit quantization for the model (QLoRA)
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)

print(f"Successfully loaded the model {model_name}!")

In [ ]:
#defining LoRA configuration

lora_config = LoraConfig(
    #rank and target_modules expanded due to stylistic nature of guided meditation
    r=16,                      
    lora_alpha=32,           
    lora_dropout=0.05,         
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",   #full attention block
        "gate_proj", "up_proj", "down_proj",      #MLP layers
    ],
    bias="none",
    task_type=TaskType.CAUSAL_LM,  
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
#tokenizing and preparing the dataset for training

from tokenizer import generate_and_tokenize_prompt

tokenized_train_dataset = train_ds.map(generate_and_tokenize_prompt)
tokenized_val_dataset = val_ds.map(generate_and_tokenize_prompt)
tokenized_test_dataset = test_ds.map(generate_and_tokenize_prompt)

#testing
print("Target Sentence: " + tokenized_test_dataset[1]['target'])
print("Meaning Representation: " + tokenized_test_dataset[1]['meaning_representation'] + "\n")

In [ ]:
#evaluating the model on the test set before training
eval_prompt = "I want to meditate on the concept of impermanence. Please guide me through a meditation session that helps me understand and accept the transient nature of life."

model_input = tokenizer(eval_prompt, return_tensors="pt").to(device)
model.eval()
with torch.no_grad():
    print(tokenizer.decode(model.generate(**model_input, max_new_tokens=256, pad_token_id=2)[0], skip_special_tokens=True))

In [ ]:
#fine-tuning the model with QLoRA

project = "OmBot-Meditation-LLM"
base_model_name = "phi-3-mini-4k-instruct"
run_name = base_model_name + "-" + project
output_dir = "./" + run_name


tokenizer.pad_token = tokenizer.eos_token


trainer = transformers.Trainer(
    model=model,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_val_dataset,
    args=transformers.TrainingArguments(
        output_dir=output_dir,
        warmup_steps=5,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        max_steps=600,         # Add a more higher value for better performance
        learning_rate=2.5e-5, # Want about 10x smaller than the Mistral learning rate
        logging_steps=50,
        #bf16=True,                #bf16 doesnot support v100,p100. For TPU, set this to true
        fp16=True,
        optim="paged_adamw_8bit",
        logging_dir="./logs",        # Directory for storing logs
        save_strategy="steps",       # Save the model checkpoint every logging step
        save_steps=50,                # Save checkpoints every 50 steps
        evaluation_strategy="steps", # Evaluate the model every logging step
        eval_steps=50,               # Evaluate and save checkpoints every 50 steps
        do_eval=True,                # Perform evaluation at the end of training
        report_to='none',           # set to 'wandb' for weights & baises logging
        run_name=f"{run_name}-{datetime.now().strftime('%Y-%m-%d-%H-%M')}",          # Name of the W&B run (optional)
    ),
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False),
)
model.config.use_cache = False  # silence the warnings. Please re-enable for inference!
trainer.train()